<a href="https://colab.research.google.com/github/umyunsang/edu/blob/main/ComputerScience/03_ai-ml-data/quantum-ml/1.quantum-ml-overview/hadamard-gate/2_hadamard_shots_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 2. Hadamard Measurement Shots Practice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umyunsang/edu/blob/main/ComputerScience/03_ai-ml-data/quantum-ml/1.quantum-ml-overview/hadamard-gate/2_hadamard_shots_practice.ipynb)

이 노트북은 `2-1.py`, `2-2.py`, `2-3.py` 실습 코드를 하나의 흐름으로 정리한 버전입니다.

**확인한 원본 코드**
- 세 파일 모두 `QuantumCircuit(1)`을 만들고 `H` gate를 적용한 뒤 측정합니다.
- 세 파일 모두 `AerSimulator`로 회로를 실행하고 `get_counts()` 결과를 출력합니다.
- 차이는 `shots` 값뿐입니다: `2-1.py = 10`, `2-2.py = 100`, `2-3.py = 1000`.

**학습 목표**
- Hadamard gate가 `|0>`을 `|0>`과 `|1>`이 거의 반반 나오는 상태로 바꾸는지 확인합니다.
- `shots`가 커질수록 측정 비율이 이론값 `0.5 / 0.5`에 가까워지는 현상을 관찰합니다.
- 같은 회로를 반복 코드 없이 재사용하는 실습 형태로 정리합니다.


## Outline

1. 원본 실습 코드의 차이 확인
2. Qiskit import와 실행 환경 확인
3. Hadamard 측정 회로 만들기
4. `shots=10`, `100`, `1000` 결과 비교
5. 측정 비율 시각화
6. 연습 문제


## 1. Source Summary

세 파일은 같은 회로를 서로 다른 반복 횟수로 실행합니다. 따라서 노트북에서는 `shots` 값만 리스트로 관리하고, 회로 생성과 실행 코드는 한 번만 정의합니다.


In [ ]:
SOURCE_SHOTS = {
    "2-1.py": 10,
    "2-2.py": 100,
    "2-3.py": 1000,
}

for source_name, shots in SOURCE_SHOTS.items():
    print(f"{source_name}: shots={shots}")


## 2. Imports

이 실습은 Qiskit과 Qiskit Aer가 필요합니다. 새 Colab 런타임에서 import가 실패하면 아래 설치 줄의 주석을 해제해 한 번 실행한 뒤, 런타임을 다시 시작하고 이어서 실행합니다.


In [ ]:
# If Qiskit is missing in a fresh Colab runtime, uncomment and run once.
# %pip install -q qiskit qiskit-aer

from __future__ import annotations

from collections.abc import Mapping

import matplotlib.pyplot as plt

try:
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator
except ModuleNotFoundError as error:
    missing_package = error.name or "qiskit"
    install_command = "%pip install -q qiskit qiskit-aer"
    raise ModuleNotFoundError(
        f"Missing {missing_package}. In Colab, run `{install_command}`, "
        "restart the runtime, and run this notebook again."
    ) from error

SHOTS_LIST = [10, 100, 1000]
SEED_SIMULATOR = 42


## 3. Build the Hadamard Measurement Circuit

초기 상태 `|0>`에 Hadamard gate를 적용하면 이론적으로 `|0>`과 `|1>`의 측정 확률이 각각 50%가 됩니다.


In [ ]:
def build_hadamard_measurement_circuit() -> QuantumCircuit:
    circuit = QuantumCircuit(1)
    circuit.h(0)
    circuit.measure_all()
    return circuit


circuit = build_hadamard_measurement_circuit()
print(circuit.draw(output="text"))


## 4. Run the Three Original Shot Counts

원본 파일 세 개의 핵심 차이인 `shots=10`, `100`, `1000`을 한 번에 비교합니다. 시뮬레이터 seed를 고정해 노트북을 다시 실행해도 같은 결과를 확인할 수 있게 했습니다.


In [ ]:
def run_counts(circuit: QuantumCircuit, shots: int, seed: int) -> dict[str, int]:
    simulator = AerSimulator(seed_simulator=seed)
    job = simulator.run(circuit, shots=shots)
    result = job.result()
    counts = result.get_counts()
    return {state: int(count) for state, count in counts.items()}


shot_counts = {
    shots: run_counts(circuit, shots, SEED_SIMULATOR + shots)
    for shots in SHOTS_LIST
}

for shots, counts in shot_counts.items():
    print(f"shots={shots}: {counts}")


## 5. Compare Measured Probabilities

Hadamard gate의 이론값은 `P(0)=0.5`, `P(1)=0.5`입니다. `shots`가 작으면 우연한 흔들림이 크고, `shots`가 커질수록 두 비율이 0.5에 가까워집니다.


In [ ]:
def normalize_counts(counts: Mapping[str, int], shots: int) -> dict[str, float]:
    return {
        "0": counts.get("0", 0) / shots,
        "1": counts.get("1", 0) / shots,
    }


probabilities = {
    shots: normalize_counts(counts, shots)
    for shots, counts in shot_counts.items()
}

print("shots | P(0)  | P(1)  | max error from 0.5")
print("------|-------|-------|-------------------")
for shots, probs in probabilities.items():
    max_error = max(abs(probs["0"] - 0.5), abs(probs["1"] - 0.5))
    print(f"{shots:>5} | {probs['0']:.3f} | {probs['1']:.3f} | {max_error:.3f}")


## 6. Visualize the Sampling Effect

각 막대가 0.5 선에 가까워지는지 확인합니다. 완전히 같지 않은 이유는 측정이 확률적 샘플링이기 때문입니다.


In [ ]:
fig, axes = plt.subplots(1, len(SHOTS_LIST), figsize=(12, 3), sharey=True)

for axis, shots in zip(axes, SHOTS_LIST, strict=True):
    probs = probabilities[shots]
    axis.bar(["0", "1"], [probs["0"], probs["1"]])
    axis.axhline(0.5, color="red", linestyle="--", linewidth=1)
    axis.set_title(f"shots={shots}")
    axis.set_ylim(0, 1)
    axis.set_xlabel("measured bit")

axes[0].set_ylabel("probability")
fig.suptitle("Hadamard measurement probabilities")
fig.tight_layout()
plt.show()


## 7. Exercise

`shots=5000`으로 같은 회로를 실행해 보세요. 결과가 `0.5 / 0.5`에 더 가까워지는지 확인합니다.


In [ ]:
exercise_shots = 5000
exercise_counts = run_counts(circuit, exercise_shots, SEED_SIMULATOR + exercise_shots)
exercise_probabilities = normalize_counts(exercise_counts, exercise_shots)

print(f"shots={exercise_shots}: {exercise_counts}")
print(f"P(0)={exercise_probabilities['0']:.3f}, P(1)={exercise_probabilities['1']:.3f}")


## Pitfall and Extension

**주의할 점**: `shots`는 gate를 여러 번 적용한다는 뜻이 아니라, 같은 회로를 같은 조건에서 여러 번 측정한다는 뜻입니다.

**확장 실습**: `circuit.h(0)`을 `circuit.x(0)` 또는 `circuit.z(0)`과 조합해 보고, 측정 확률이 어떻게 달라지는지 비교해 보세요.
